In [ ]:
# 🔹 Gerekli kütüphaneler
import numpy as np
import optuna
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import pandas as pd

In [ ]:
!pip install -q optuna

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive')
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")

In [ ]:
# 🔹 Veriyi yükle
df = pd.read_csv("BALANCED_DATASET.csv")  # 'text' ve 'label' sütunları olmalı
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

# 🔹 Model listesi
model_names = [
    "distilbert-base-uncased",
    "microsoft/deberta-v3-base",
    "google/electra-base-discriminator",
    "albert-base-v2"
]

# 🔹 Veri seti yükle
df = pd.read_csv("BALANCED_DATASET.csv")  # 'text' ve 'label' sütunları olmalı
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Tokenization fonksiyonu
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

# 🔹 Değerlendirme fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔁 Tüm modeller için döngü
for model_name in model_names:
    print(f"\n🔍 {model_name} için Optuna başlatılıyor...\n")

    # Tokenizer ve veri tokenizasyonu
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized = dataset.map(tokenize_fn, batched=True)

    # Modeli yeniden oluşturacak fonksiyon
    def model_init():
        return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    # Optuna'nın deneyeceği hiperparametre alanı
    def hp_space(trial):
        return {
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
            "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
            "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
            "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        }

    # Eğitim parametreleri (sabit olanlar)
    training_args = TrainingArguments(
        output_dir=f"./optuna_{model_name.replace('/', '_')}_results",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        report_to="none",
        disable_tqdm=True
    )

    # Trainer
    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    # Optuna hiperparametre araması
    best_run = trainer.hyperparameter_search(
        direction="maximize",
        hp_space=hp_space,
        n_trials=10,
        backend="optuna"
    )

    # Sonuçları yazdır
    print(f"✅ {model_name} için en iyi sonuçlar:")
    print(f"  - En iyi f1_macro (objective): {best_run.objective:.4f}")
    print(f"  - learning_rate              : {best_run.hyperparameters['learning_rate']}")
    print(f"  - num_epochs                 : {best_run.hyperparameters['num_train_epochs']}")
    print(f"  - batch_size                 : {best_run.hyperparameters['per_device_train_batch_size']}")
    print(f"  - weight_decay               : {best_run.hyperparameters['weight_decay']}")
